In [ ]:
# Fairy2iOS — uma única célula, busca dinâmica do último build PDRK.
from pathlib import Path
import subprocess, os, sys, importlib.util, importlib, getpass
REPO = Path("/content/Fairy2iOS-repo")
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "Fairy2iOS",
                    "https://github.com/joao9vegeto-stack/KintTany.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "Fairy2iOS"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "FETCH_HEAD"], check=True)
COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
COLAB_SRC = REPO / "fairy2" / "colab"
if str(COLAB_SRC) in sys.path:
    sys.path.remove(str(COLAB_SRC))
sys.path.insert(0, str(COLAB_SRC))
importlib.invalidate_caches()
sys.modules.pop("ipa_validator", None)
script = COLAB_SRC / "transfer_ipa.py"
spec = importlib.util.spec_from_file_location("fairy2_dynamic_latest_transfer", str(script))
transfer_latest = importlib.util.module_from_spec(spec)
spec.loader.exec_module(transfer_latest)
assert Path(transfer_latest.__file__).resolve() == script.resolve()
assert transfer_latest.BRANCH == "Fairy2iOS"
assert callable(transfer_latest.select_pdrk_direct_io_artifact)
assert callable(transfer_latest.transfer_pdrk_latest)
print("[Fairy2iOS] Branch atualizada:", COMMIT, flush=True)
if not os.environ.get("FAIRY2_GITHUB_TOKEN"):
    os.environ["FAIRY2_GITHUB_TOKEN"] = getpass.getpass("GitHub PAT (Actions: Read): ").strip()
assert os.environ["FAIRY2_GITHUB_TOKEN"], "Token vazio"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "google-api-python-client", "google-auth-httplib2", "requests"], check=True)
from google.colab import auth as colab_auth
colab_auth.authenticate_user()
RESULTADO_IPA = transfer_latest.transfer_pdrk_latest()
assert RESULTADO_IPA["pdrk_direct_io_candidate"] is True
assert RESULTADO_IPA["diagnostic_only"] is False
print("RUN MAIS RECENTE:", RESULTADO_IPA["run_id"], flush=True)
print("IPA NO DRIVE:", RESULTADO_IPA["ipa_name"], flush=True)
print("SHA-256:", RESULTADO_IPA["ipa_sha256"], flush=True)
print("STATUS: CANDIDATA GLOBAL PDRK; execução real no iPhone ainda necessária.", flush=True)
